GitHumから画像をダウンロード

画像のサイズを確認しなさい　幅W　×　高さH　単位はピクセル

In [ ]:
# サンプル画像をGitHubからダウンロード

from PIL import Image
from io import BytesIO
import requests
from matplotlib import pyplot as plt

# GitHub上の画像URL
image_github = "https://github.com/Taka-Lab-Python/Human_AI_Exercises/blob/main/cat1.jpg"

# GitHubのURLをRaw URLへ変換
image_url = image_github.replace(
"github.com", "raw.githubusercontent.com").replace("/blob/", "/")

# GitHubから画像を取得
response = requests.get(image_url, timeout=30)
response.raise_for_status()
# PIL画像に変換
img_pil = Image.open(BytesIO(response.content)).convert("RGB")

print(f"画像のサイズ (幅, 高さ)ピクセル: {img_pil.size}")
plt.imshow(img_pil)  # 画像を描かせる
plt.show()

生成AIやDeep Learningに読み込ませる場合、メモリー不足にならないように縮小する

縮小した画像を確認する

In [ ]:
# 1/64に画像を縮小
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from PIL import Image

size = 64   # 1/64に縮小
# --- サイズを縮小 ---
resize = transforms.Resize((size, size))
img_small = resize(img_pil)

# 画像を表示させる
plt.imshow(img_small)
plt.show()


画像は、数値データになっていることを確認する

In [ ]:
# --- 数値データ（NumPy配列）に変換 ---
image_num = np.array(img_small)    # 0〜255の整数
print("配列の形 (高さ, 幅, チャンネル):", image_num.shape)
print("データ型:", image_num.dtype)
print("最小値:", image_num.min(), " 最大値:", image_num.max())

# ニューラルネットワークの入力形式(決まり) （PyTorchのTensor）
tensor = transforms.ToTensor()(image_num)  # 0〜1に正規化、(C, H, W)
print("Tensorの形 (Color, H, W):", tensor.shape)


配列の形 (高さ, 幅, チャンネル): (64, 64, 3)
データ型: uint8
最小値: 5  最大値: 236
Tensorの形 (Color, H, W): torch.Size([3, 64, 64])


In [ ]:
# 赤チャンネルだけ取り出してみる　（形: 64×64、値: 0〜1）
r = tensor[0].numpy()
print("形:", r.shape, " 最小:", r.min().round(3), " 最大:",r.max().round(3))


形: (64, 64)  最小: 0.02  最大: 0.925


In [ ]:
#  R(赤)のヒートマップを描かせる
import seaborn as sns
import matplotlib.pyplot as plt

red_pic = tensor[0].numpy()  # Rチャンネル（64×64、0〜1）

plt.figure(figsize=(14, 16))
sns.heatmap(
    red_pic,
    cmap="Reds",          #  "viridis", "coolwarm" なども可
    vmin=0, vmax=1,
    annot=True,           # マス目に数値を書き込む
    fmt=".2f",            # 小数2桁
    annot_kws={"size": 5},
    square=True,          # マスを正方形に
    cbar_kws={"shrink": 0.8},
    xticklabels=8, yticklabels=8  # 目盛りは8ピクセルごと
)
plt.show()

R・G・Bの3枚のマップを重ねて、カラー画像に戻すコード。

In [ ]:
# 3枚のマップ（それぞれ64×64、値は0〜1）
r = tensor[0].numpy()
g = tensor[1].numpy()
b = tensor[2].numpy()

zero = np.zeros_like(r)
steps = [
    ("R only",    np.stack([r,    zero, zero], axis=2)),
    ("R + G",     np.stack([r,    g,    zero], axis=2)),
    ("R + G + B", np.stack([r,    g,    b   ], axis=2)),
]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (title, im) in zip(ax, steps):
    a.imshow(im, interpolation="nearest"); a.set_title(title); a.axis("off")
plt.show()

In [ ]:
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 64×64に縮小して、0〜1の数値に変換
img_64 = transforms.Resize((64, 64))(img_pil)
T = transforms.ToTensor()(img_64)       # テンソル：形 (3, 64, 64)

def show(x):
    """(3, 64, 64) を表示用の (64, 64, 3) に並べ替える"""
    return x.permute(1, 2, 0).numpy()

In [ ]:
R = T[0]                                  # 行列：形 (64, 64)
R_bright = torch.clamp(R * 1.5, 0, 1)
R_dark   = R * 0.5

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, m, t in zip(ax, [R, R_bright, R_dark], ["Original R", "R x1.5", "R x0.5"]):
    a.imshow(m.numpy(), cmap="Reds", vmin=0, vmax=1); a.set_title(t); a.axis("off")
plt.suptitle("Matrix: R channel 64x64")
plt.show()